# KAIS Research Software Classification — complete experiment runner

This notebook is designed for the September 2026 supervisor-review version of the journal paper.

**It runs and checkpoints:**
1. Dataset inventory/audit from the existing Zenodo replication record.
2. Single-label TF-IDF experiments across Logistic Regression, Linear SVM, Random Forest and XGBoost.
3. Fair matched-cohort comparisons using exactly the same samples/folds across attributes.
4. Attribute-combination experiments requested by the previous reviewers.
5. Multi-label classification with iterative stratification.
6. Sentence-BERT experiments using the same model used in the earlier work (`all-MiniLM-L6-v2`).
7. Independent-domain external validation on bio.tools / EDAM topics.
8. An optional lightweight ~3B LLM pilot when a GPU is available.

All results are written to Google Drive after every experiment family, so a Colab disconnect does not destroy completed work.

### Before running
In Colab choose **Runtime → Change runtime type → GPU**. Then run all cells. The first Drive mount requires one Google authorization click.

The notebook intentionally uses fixed random seeds and fits text vectorizers **inside each training fold** to avoid leakage.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, json, random, re, ast, math, time, warnings
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/phd_kais_experiments_2026_09_01')
DATA_DIR = ROOT / 'data'
RESULTS_DIR = ROOT / 'results'
CACHE_DIR = ROOT / 'cache'
for p in [ROOT, DATA_DIR, RESULTS_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)

# Core runs are ON. The LLM pilot is optional and can be switched off if the core paper
# experiments need to finish first.
RUN_XGBOOST = True
RUN_SBERT = True
RUN_BIOTOOLS = True
RUN_LLM_PILOT = True

print('Output directory:', ROOT)


In [ ]:
!pip -q install -U scikit-learn pandas numpy scipy requests tqdm iterative-stratification xgboost sentence-transformers transformers accelerate bitsandbytes

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
from scipy import stats

import sklearn
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    classification_report, hamming_loss
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier

from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

warnings.filterwarnings('ignore')
print('scikit-learn:', sklearn.__version__)


## 1. Download the existing replication dataset

The previous MSR review identifies Zenodo record **17345363** as the dataset artifact. This cell downloads every file from that record through the Zenodo REST API and preserves the originals in Drive.


In [ ]:
ZENODO_RECORD = 17345363
ZENODO_DIR = DATA_DIR / f'zenodo_{ZENODO_RECORD}'
ZENODO_DIR.mkdir(exist_ok=True)

record_url = f'https://zenodo.org/api/records/{ZENODO_RECORD}'
r = requests.get(record_url, timeout=60)
r.raise_for_status()
record = r.json()

print('Zenodo title:', record.get('metadata', {}).get('title'))
files = record.get('files', [])
print('Files:', len(files))

manifest = []
for f in files:
    key = f.get('key') or f.get('filename')
    links = f.get('links', {})
    url = links.get('content') or links.get('self')
    out = ZENODO_DIR / key
    manifest.append({'name': key, 'size': f.get('size'), 'url': url, 'path': str(out)})
    if out.exists() and out.stat().st_size > 0:
        print('cached:', key)
        continue
    print('downloading:', key)
    with requests.get(url, stream=True, timeout=120) as resp:
        resp.raise_for_status()
        with open(out, 'wb') as fh:
            for chunk in resp.iter_content(chunk_size=1024*1024):
                if chunk:
                    fh.write(chunk)

pd.DataFrame(manifest).to_csv(RESULTS_DIR / 'zenodo_manifest.csv', index=False)
display(pd.DataFrame(manifest))


## 2. Inventory all tabular artifacts and automatically identify the widest PwC classification table

The old artifact changed structure across revisions. Instead of hard-coding one filename, this loader scans CSV/JSON/Parquet/Excel files and scores them by whether they contain category labels plus the attributes needed by the paper.


In [ ]:
import zipfile, shutil

# Expand archives without overwriting originals.
for z in ZENODO_DIR.rglob('*'):
    if z.is_file() and z.suffix.lower() == '.zip':
        target = ZENODO_DIR / ('unzipped_' + z.stem)
        if not target.exists():
            target.mkdir()
            with zipfile.ZipFile(z) as zz:
                zz.extractall(target)

def norm_col(c):
    return re.sub(r'[^a-z0-9]+', '_', str(c).strip().lower()).strip('_')

LABEL_NAMES = [
    'labels','label','categories','category','research_area','research_areas',
    'area','areas','task','tasks','research_category','research_categories'
]
ID_NAMES = [
    'sample_id','id','paper_id','publication_id','repo_id','repository_id',
    'github_url','repository_url','repo_url','url','paper_url','publication_url'
]
ATTR_ALIASES = {
    'abstract': ['abstract','paper_abstract','publication_abstract','reference_abstract','research_publication_abstract'],
    'paper_title': ['paper_title','publication_title','reference_title','research_publication_title'],
    'readme': ['readme','readme_text','repository_readme','repo_readme','github_readme'],
    'description': ['description','repo_description','repository_description','github_description','somef_description','software_description'],
    'repo_title': ['repo_title','repository_title','github_title','repo_name','repository_name','github_name'],
    'keywords': ['keywords','topics','github_topics','repo_topics','repository_topics','github_keywords']
}

def read_table(path):
    suf = path.suffix.lower()
    try:
        if suf == '.csv':
            return pd.read_csv(path, low_memory=False)
        if suf in ['.parquet','.pq']:
            return pd.read_parquet(path)
        if suf in ['.xlsx','.xls']:
            return pd.read_excel(path)
        if suf in ['.json','.jsonl']:
            try:
                return pd.read_json(path, lines=(suf=='.jsonl'))
            except Exception:
                obj = json.load(open(path))
                if isinstance(obj, list):
                    return pd.json_normalize(obj)
                if isinstance(obj, dict):
                    for k,v in obj.items():
                        if isinstance(v, list) and len(v) > 20:
                            return pd.json_normalize(v)
    except Exception as e:
        print('skip', path.name, type(e).__name__, str(e)[:120])
    return None

tables = []
for p in ZENODO_DIR.rglob('*'):
    if not p.is_file() or p.suffix.lower() not in ['.csv','.json','.jsonl','.parquet','.pq','.xlsx','.xls']:
        continue
    df = read_table(p)
    if df is None or len(df) < 20:
        continue
    df.columns = [norm_col(c) for c in df.columns]
    cols = set(df.columns)
    label_hits = [c for c in LABEL_NAMES if c in cols]
    attr_hits = {a: next((c for c in aliases if c in cols), None) for a,aliases in ATTR_ALIASES.items()}
    attr_hits = {k:v for k,v in attr_hits.items() if v}
    id_hits = [c for c in ID_NAMES if c in cols]
    score = 10*len(attr_hits) + 5*bool(label_hits) + 2*bool(id_hits) + math.log10(max(len(df),1))
    tables.append({
        'path': str(p), 'name': p.name, 'rows': len(df), 'cols': len(df.columns),
        'label_hits': label_hits, 'id_hits': id_hits, 'attr_hits': attr_hits, 'score': score,
        '_df': df
    })

inventory = pd.DataFrame([{k:v for k,v in t.items() if k != '_df'} for t in tables]).sort_values('score', ascending=False)
inventory.to_csv(RESULTS_DIR / 'dataset_inventory.csv', index=False)
display(inventory.head(30))

eligible = [t for t in tables if t['label_hits'] and t['attr_hits']]
if not eligible:
    raise RuntimeError('No classification table with both labels and text attributes was found in the Zenodo artifact.')

# Widest candidate first; number of supported attributes dominates, then row count.
eligible.sort(key=lambda t: (len(t['attr_hits']), t['rows']), reverse=True)
chosen = eligible[0]
raw = chosen['_df'].copy()
print('\nSelected table:', chosen['path'])
print('rows:', len(raw), 'attributes:', chosen['attr_hits'], 'label:', chosen['label_hits'][0])
display(raw.head(3))


## 3. Standardize labels and attributes

The six Papers-with-Code research-area categories used in the existing paper are retained. Multi-label samples are *not* thrown away globally: we derive a single-label view for the multiclass experiments and retain one-or-more-label samples for the multilabel experiments.


In [ ]:
TARGET_LABELS = [
    'Computer Vision',
    'Natural Language Processing',
    'Graphs',
    'Reinforcement Learning',
    'Sequential',
    'Audio'
]

LABEL_ALIASES = {
    'computer vision':'Computer Vision', 'cv':'Computer Vision',
    'natural language processing':'Natural Language Processing', 'nlp':'Natural Language Processing',
    'graphs':'Graphs', 'graph':'Graphs', 'graph learning':'Graphs',
    'reinforcement learning':'Reinforcement Learning', 'rl':'Reinforcement Learning',
    'sequential':'Sequential', 'sequence':'Sequential',
    'audio':'Audio', 'speech':'Audio'
}

def parse_labels(v):
    if v is None or (isinstance(v,float) and np.isnan(v)):
        return []
    if isinstance(v, (list,tuple,set,np.ndarray)):
        vals = list(v)
    else:
        s = str(v).strip()
        if not s:
            return []
        vals = None
        for parser in [ast.literal_eval, json.loads]:
            try:
                x = parser(s)
                if isinstance(x, (list,tuple,set)):
                    vals = list(x); break
            except Exception:
                pass
        if vals is None:
            vals = re.split(r'\s*[|;,]\s*', s)
    out = []
    for x in vals:
        if isinstance(x, dict):
            x = x.get('name') or x.get('term') or x.get('label') or ''
        key = str(x).strip().lower()
        canonical = LABEL_ALIASES.get(key)
        if canonical and canonical not in out:
            out.append(canonical)
    return out

label_col = chosen['label_hits'][0]
attr_map = chosen['attr_hits']

df = pd.DataFrame(index=raw.index)
df['labels'] = raw[label_col].map(parse_labels)

# Stable ID for auditing. Prefer provider ID; otherwise hash the row index + available title/url.
id_col = chosen['id_hits'][0] if chosen['id_hits'] else None
if id_col:
    df['sample_id'] = raw[id_col].astype(str)
else:
    df['sample_id'] = raw.index.astype(str)

for canonical, source in attr_map.items():
    df[canonical] = raw[source].fillna('').astype(str).str.strip()

# If the source has a generic title column and no explicit publication/repository title,
# only use it when its meaning is clear from the filename.
if 'title' in raw.columns:
    fname = chosen['name'].lower()
    if 'paper_title' not in df and any(k in fname for k in ['paper','publication','abstract']):
        df['paper_title'] = raw['title'].fillna('').astype(str)
    elif 'repo_title' not in df and any(k in fname for k in ['repo','software','github']):
        df['repo_title'] = raw['title'].fillna('').astype(str)

df = df[df['labels'].map(len) > 0].drop_duplicates('sample_id').reset_index(drop=True)

# Ensure all expected columns exist for uniform code.
for c in ['abstract','paper_title','readme','description','repo_title','keywords']:
    if c not in df:
        df[c] = ''

df.to_parquet(DATA_DIR / 'pwc_standardized.parquet', index=False)

print('Usable labeled rows:', len(df))
print('Single-label:', (df.labels.map(len)==1).sum())
print('Multi-or-single-label:', len(df))
print('\nLabel incidence:')
for lab in TARGET_LABELS:
    print(f'{lab:30s}', df.labels.map(lambda x: lab in x).sum())

print('\nAttribute completeness:')
for c in ['abstract','paper_title','readme','description','repo_title','keywords']:
    print(f'{c:15s}', (df[c].str.len()>0).sum())

audit = []
for c in ['abstract','paper_title','readme','description','repo_title','keywords']:
    audit.append({'attribute':c,'nonempty':int((df[c].str.len()>0).sum()),'total':len(df),'coverage':float((df[c].str.len()>0).mean())})
pd.DataFrame(audit).to_csv(RESULTS_DIR/'pwc_attribute_coverage.csv',index=False)
display(pd.DataFrame(audit))


## 4. Shared experiment utilities

All vectorizers are fitted within each training fold. The same fixed fold assignment is reused for every model within a cohort. Results include macro-, micro-, and weighted-F1, mean, standard deviation and 95% confidence interval.


In [ ]:
def join_fields(frame, fields):
    fields = [f for f in fields if f in frame.columns]
    if not fields:
        return pd.Series(['']*len(frame), index=frame.index)
    parts = []
    for f in fields:
        vals = frame[f].fillna('').astype(str)
        parts.append(('[['+f.upper()+']] ') + vals)
    out = parts[0]
    for p in parts[1:]:
        out = out + '\n' + p
    return out.str.strip()

def ci95(values):
    a = np.asarray(values, dtype=float)
    if len(a) < 2:
        return np.nan
    return float(stats.t.ppf(0.975, len(a)-1) * a.std(ddof=1) / np.sqrt(len(a)))

def model_factories():
    m = {
        'logreg': lambda: LogisticRegression(max_iter=3000, class_weight='balanced', random_state=SEED),
        'linear_svm': lambda: LinearSVC(class_weight='balanced', random_state=SEED),
        'random_forest': lambda: RandomForestClassifier(
            n_estimators=300, min_samples_leaf=1, class_weight='balanced_subsample',
            n_jobs=-1, random_state=SEED
        )
    }
    if RUN_XGBOOST:
        from xgboost import XGBClassifier
        m['xgboost'] = lambda: XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.08,
            subsample=0.85, colsample_bytree=0.85,
            tree_method='hist', n_jobs=-1, random_state=SEED,
            eval_metric='mlogloss'
        )
    return m

def cv_single_label(frame, fields, model_name, n_splits=5):
    work = frame.copy()
    text = join_fields(work, fields)
    work = work[text.str.len() > 0].copy()
    work['_text'] = text.loc[work.index]
    work['_label'] = work.labels.map(lambda x: x[0])
    # Only valid six-area labels.
    work = work[work['_label'].isin(TARGET_LABELS)].reset_index(drop=True)

    y = work['_label'].to_numpy()
    X = work['_text'].to_numpy()
    le = LabelEncoder().fit(y)
    y_enc = le.transform(y)

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    fold_rows, per_class_rows = [], []
    factory = model_factories()[model_name]

    for fold,(tr,te) in enumerate(cv.split(X,y_enc),1):
        vec = TfidfVectorizer(
            lowercase=True, strip_accents='unicode', ngram_range=(1,2),
            min_df=2, max_df=0.98, sublinear_tf=True, max_features=100000
        )
        Xtr = vec.fit_transform(X[tr])
        Xte = vec.transform(X[te])
        model = factory()
        model.fit(Xtr, y_enc[tr])
        pred = model.predict(Xte)

        metrics = {
            'fold': fold,
            'f1_macro': f1_score(y_enc[te],pred,average='macro',zero_division=0),
            'f1_micro': f1_score(y_enc[te],pred,average='micro',zero_division=0),
            'f1_weighted': f1_score(y_enc[te],pred,average='weighted',zero_division=0),
            'accuracy': accuracy_score(y_enc[te],pred)
        }
        fold_rows.append(metrics)
        report = classification_report(y_enc[te],pred,target_names=le.classes_,output_dict=True,zero_division=0)
        for lab in le.classes_:
            per_class_rows.append({'fold':fold,'label':lab,'f1':report[lab]['f1-score'],'support':report[lab]['support']})

    folds = pd.DataFrame(fold_rows)
    summary = {
        'model':model_name, 'fields':'+'.join(fields), 'n':len(work), 'folds':n_splits
    }
    for metric in ['f1_macro','f1_micro','f1_weighted','accuracy']:
        summary[metric+'_mean'] = folds[metric].mean()
        summary[metric+'_std'] = folds[metric].std(ddof=1)
        summary[metric+'_ci95'] = ci95(folds[metric])
    return summary, folds, pd.DataFrame(per_class_rows)

def save_checkpoint(name, rows):
    out = RESULTS_DIR / name
    pd.DataFrame(rows).to_csv(out, index=False)
    print('saved', out)
    return out


## 5. Single-label robustness across classifiers — available-case analysis

This answers the previous-review concern that the conclusions might depend on Random Forest. Each attribute uses all rows where that attribute is available, and all classifiers use the same five stratified folds for that attribute.


In [ ]:
single = df[df.labels.map(len)==1].copy().reset_index(drop=True)

INDIVIDUAL_CONFIGS = {
    'abstract':['abstract'],
    'paper_title':['paper_title'],
    'readme':['readme'],
    'description':['description'],
    'repo_title':['repo_title'],
    'keywords':['keywords']
}

single_model_rows = []
single_perclass = []

for cfg,fields in INDIVIDUAL_CONFIGS.items():
    if (join_fields(single,fields).str.len()>0).sum() < 50:
        print('skip sparse config', cfg); continue
    for model_name in model_factories():
        print('\nRUN', cfg, model_name)
        summary, folds, perclass = cv_single_label(single, fields, model_name)
        summary['config'] = cfg
        single_model_rows.append(summary)
        perclass['config'] = cfg
        perclass['model'] = model_name
        single_perclass.append(perclass)
        save_checkpoint('single_label_available_case.csv', single_model_rows)

single_available = pd.DataFrame(single_model_rows)
single_available.to_csv(RESULTS_DIR/'single_label_available_case.csv', index=False)
if single_perclass:
    pd.concat(single_perclass,ignore_index=True).to_csv(RESULTS_DIR/'single_label_per_class.csv',index=False)

display(single_available.sort_values('f1_macro_mean',ascending=False))


## 6. Matched-cohort and attribute-combination experiments

The old paper compared attributes on datasets of different sizes. Here we build the largest defensible common cohort and compare attributes/combinations on **the same software items and same folds**.

The combination experiment directly tests the reviewer request: does combining publication and repository metadata actually improve performance?


In [ ]:
available_attrs = [c for c in ['abstract','paper_title','readme','description','repo_title','keywords']
                   if (single[c].str.len()>0).sum() >= 100]

# Start with core attributes and iteratively avoid collapsing to an unusably tiny cohort.
priority = [c for c in ['abstract','readme','description','repo_title','keywords','paper_title'] if c in available_attrs]
matched_fields = []
matched = single.copy()
for c in priority:
    candidate = matched[matched[c].str.len()>0]
    # Require enough rows and at least 5 examples in every represented target class for 5-fold CV.
    counts = candidate.labels.map(lambda x:x[0]).value_counts()
    if len(candidate) >= 300 and len(counts) >= 4 and counts.min() >= 5:
        matched = candidate.copy()
        matched_fields.append(c)

matched = matched.reset_index(drop=True)
print('Matched fields:', matched_fields)
print('Matched cohort size:', len(matched))
display(matched.labels.map(lambda x:x[0]).value_counts())

# Save exact cohort for reproducibility.
matched[['sample_id','labels']+matched_fields].to_parquet(DATA_DIR/'pwc_matched_cohort.parquet',index=False)

# Compare individual fields with a strong linear baseline to isolate the attribute.
matched_rows = []
for c in matched_fields:
    summary,_,_ = cv_single_label(matched,[c],'linear_svm')
    summary['config']=c
    matched_rows.append(summary)
save_checkpoint('single_label_matched_individual.csv',matched_rows)

combo_configs = {}
if 'abstract' in matched_fields:
    combo_configs['abstract']=['abstract']
repo_fields = [c for c in ['repo_title','keywords','description','readme'] if c in matched_fields]
if repo_fields:
    combo_configs['repository_all']=repo_fields
if 'repo_title' in matched_fields and 'keywords' in matched_fields:
    combo_configs['repo_title_keywords']=['repo_title','keywords']
if 'abstract' in matched_fields and repo_fields:
    combo_configs['abstract_repository_all']=['abstract']+repo_fields
if 'abstract' in matched_fields and 'description' in matched_fields:
    combo_configs['abstract_description']=['abstract','description']

combo_rows=[]
for name,fields in combo_configs.items():
    print('RUN COMBINATION',name,fields)
    summary,_,_=cv_single_label(matched,fields,'linear_svm')
    summary['config']=name
    combo_rows.append(summary)
    save_checkpoint('single_label_attribute_combinations.csv',combo_rows)

matched_results=pd.DataFrame(matched_rows)
combo_results=pd.DataFrame(combo_rows)
display(matched_results.sort_values('f1_macro_mean',ascending=False))
display(combo_results.sort_values('f1_macro_mean',ascending=False))


## 7. Multi-label experiment

Single-label samples remain valid multi-label observations. We keep every row with one or more target labels, use iterative multilabel stratification, and report micro/macro/weighted F1, subset accuracy and Hamming loss.


In [ ]:
def cv_multilabel(frame, fields, n_splits=5):
    work=frame.copy()
    txt=join_fields(work,fields)
    work=work[txt.str.len()>0].copy()
    work['_text']=txt.loc[work.index]
    work=work[work.labels.map(len)>0].reset_index(drop=True)

    mlb=MultiLabelBinarizer(classes=TARGET_LABELS)
    Y=mlb.fit_transform(work.labels)
    X=work['_text'].to_numpy()

    cv=MultilabelStratifiedKFold(n_splits=n_splits,shuffle=True,random_state=SEED)
    rows=[]
    for fold,(tr,te) in enumerate(cv.split(X,Y),1):
        vec=TfidfVectorizer(lowercase=True,strip_accents='unicode',ngram_range=(1,2),
                            min_df=2,max_df=.98,sublinear_tf=True,max_features=100000)
        Xtr=vec.fit_transform(X[tr]); Xte=vec.transform(X[te])
        base=LogisticRegression(max_iter=3000,class_weight='balanced',random_state=SEED)
        model=OneVsRestClassifier(base,n_jobs=-1)
        model.fit(Xtr,Y[tr])
        pred=model.predict(Xte)
        rows.append({
            'fold':fold,
            'f1_macro':f1_score(Y[te],pred,average='macro',zero_division=0),
            'f1_micro':f1_score(Y[te],pred,average='micro',zero_division=0),
            'f1_weighted':f1_score(Y[te],pred,average='weighted',zero_division=0),
            'subset_accuracy':accuracy_score(Y[te],pred),
            'hamming_loss':hamming_loss(Y[te],pred)
        })
    folds=pd.DataFrame(rows)
    summary={'fields':'+'.join(fields),'n':len(work),'folds':n_splits}
    for metric in ['f1_macro','f1_micro','f1_weighted','subset_accuracy','hamming_loss']:
        summary[metric+'_mean']=folds[metric].mean()
        summary[metric+'_std']=folds[metric].std(ddof=1)
        summary[metric+'_ci95']=ci95(folds[metric])
    return summary,folds

multi_configs={}
for c in ['abstract','readme','description','keywords']:
    if (df[c].str.len()>0).sum()>=100:
        multi_configs[c]=[c]
if repo_fields:
    multi_configs['repository_all']=repo_fields
if 'abstract' in df and repo_fields:
    multi_configs['abstract_repository_all']=['abstract']+repo_fields

multi_rows=[]
for name,fields in multi_configs.items():
    print('RUN MULTILABEL',name)
    summary,_=cv_multilabel(df,fields)
    summary['config']=name
    multi_rows.append(summary)
    save_checkpoint('multilabel_tfidf.csv',multi_rows)

multi_results=pd.DataFrame(multi_rows)
display(multi_results.sort_values('f1_micro_mean',ascending=False))


## 8. Sentence-BERT comparison

This uses the same Sentence-BERT checkpoint as the earlier work: **all-MiniLM-L6-v2**. Embeddings are cached in Drive, then evaluated with five identical stratified folds using Linear SVM so the representation is the varying factor.


In [ ]:
if RUN_SBERT:
    import torch
    from sentence_transformers import SentenceTransformer
    from sklearn.svm import LinearSVC

    device='cuda' if torch.cuda.is_available() else 'cpu'
    print('SBERT device:',device)
    sbert=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',device=device)

    def sbert_embed(name,texts):
        path=CACHE_DIR/f'sbert_{name}_{len(texts)}.npy'
        if path.exists():
            return np.load(path)
        emb=sbert.encode(list(texts),batch_size=64,show_progress_bar=True,
                         convert_to_numpy=True,normalize_embeddings=True)
        np.save(path,emb)
        return emb

    def cv_dense_single(frame,fields,name):
        work=frame.copy()
        txt=join_fields(work,fields)
        work=work[txt.str.len()>0].copy()
        work['_text']=txt.loc[work.index]
        work['_label']=work.labels.map(lambda x:x[0])
        work=work[work['_label'].isin(TARGET_LABELS)].reset_index(drop=True)
        X=sbert_embed(name,work['_text'])
        le=LabelEncoder().fit(work['_label'])
        y=le.transform(work['_label'])
        cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
        rows=[]
        for fold,(tr,te) in enumerate(cv.split(X,y),1):
            model=LinearSVC(class_weight='balanced',random_state=SEED)
            model.fit(X[tr],y[tr]); pred=model.predict(X[te])
            rows.append({
                'fold':fold,
                'f1_macro':f1_score(y[te],pred,average='macro',zero_division=0),
                'f1_micro':f1_score(y[te],pred,average='micro',zero_division=0),
                'f1_weighted':f1_score(y[te],pred,average='weighted',zero_division=0),
                'accuracy':accuracy_score(y[te],pred)
            })
        folds=pd.DataFrame(rows)
        summary={'config':name,'fields':'+'.join(fields),'n':len(work),'embedding':'all-MiniLM-L6-v2'}
        for m in ['f1_macro','f1_micro','f1_weighted','accuracy']:
            summary[m+'_mean']=folds[m].mean(); summary[m+'_std']=folds[m].std(ddof=1); summary[m+'_ci95']=ci95(folds[m])
        return summary

    sbert_configs={k:v for k,v in combo_configs.items()}
    # Also add individual matched fields.
    for c in matched_fields:
        sbert_configs.setdefault(c,[c])

    sbert_rows=[]
    for name,fields in sbert_configs.items():
        print('RUN SBERT',name)
        sbert_rows.append(cv_dense_single(matched,fields,'matched_'+name))
        save_checkpoint('sbert_matched_results.csv',sbert_rows)
    sbert_results=pd.DataFrame(sbert_rows)
    display(sbert_results.sort_values('f1_macro_mean',ascending=False))


## 9. Independent-domain validation: bio.tools

This is deliberately treated as an **external-validity experiment**, not as if bio.tools and Papers with Code had identical taxonomies. It uses bio.tools' community-maintained EDAM topics, selects the six most frequent single-topic categories, and tests how well text metadata predicts those categories.

This answers the strongest generalizability concern from the previous reviews without pretending the ecosystems are semantically identical.


In [ ]:
if RUN_BIOTOOLS:
    BIOTOOLS_CACHE=DATA_DIR/'biotools.json'
    if BIOTOOLS_CACHE.exists():
        tools_data=json.load(open(BIOTOOLS_CACHE))
    else:
        url='https://bio.tools/api/t/?format=json'
        tools_data=[]
        pbar=tqdm(desc='bio.tools pages')
        while url:
            rr=requests.get(url,timeout=60)
            rr.raise_for_status()
            obj=rr.json()
            batch=obj.get('list') or obj.get('data') or obj.get('results') or []
            tools_data.extend(batch)
            url=obj.get('next')
            pbar.update(1)
            # Defensive bound; current registry is well below this.
            if len(tools_data)>100000:
                break
        pbar.close()
        json.dump(tools_data,open(BIOTOOLS_CACHE,'w'))

    bio_rows=[]
    for t in tools_data:
        topics=[]
        for x in (t.get('topic') or []):
            if isinstance(x,dict):
                term=x.get('term') or x.get('name')
            else:
                term=str(x)
            if term: topics.append(term.strip())
        topics=list(dict.fromkeys(topics))
        bio_rows.append({
            'id':t.get('biotoolsID') or t.get('name'),
            'name':t.get('name') or '',
            'description':t.get('description') or '',
            'topics':topics
        })
    bio=pd.DataFrame(bio_rows)
    bio=bio[bio.topics.map(len)==1].copy()
    bio['label']=bio.topics.map(lambda x:x[0])

    top6=bio.label.value_counts().head(6).index.tolist()
    bio=bio[bio.label.isin(top6)].copy().reset_index(drop=True)
    print('Top six EDAM topics:',top6)
    display(bio.label.value_counts())

    def bio_cv(field_name,text):
        work=bio.copy()
        work['_text']=text.fillna('').astype(str).str.strip()
        work=work[work._text.str.len()>0].reset_index(drop=True)
        y=work.label.to_numpy()
        cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
        rows=[]
        for fold,(tr,te) in enumerate(cv.split(work._text,y),1):
            pipe=Pipeline([
                ('tfidf',TfidfVectorizer(ngram_range=(1,2),min_df=2,max_df=.98,
                                         sublinear_tf=True,max_features=100000)),
                ('clf',LinearSVC(class_weight='balanced',random_state=SEED))
            ])
            pipe.fit(work._text.iloc[tr],y[tr]); pred=pipe.predict(work._text.iloc[te])
            rows.append({
                'fold':fold,
                'f1_macro':f1_score(y[te],pred,average='macro',zero_division=0),
                'f1_micro':f1_score(y[te],pred,average='micro',zero_division=0),
                'f1_weighted':f1_score(y[te],pred,average='weighted',zero_division=0),
                'accuracy':accuracy_score(y[te],pred)
            })
        f=pd.DataFrame(rows)
        out={'config':field_name,'n':len(work),'labels':' | '.join(top6)}
        for m in ['f1_macro','f1_micro','f1_weighted','accuracy']:
            out[m+'_mean']=f[m].mean();out[m+'_std']=f[m].std(ddof=1);out[m+'_ci95']=ci95(f[m])
        return out

    bio_exp=[
        bio_cv('description',bio.description),
        bio_cv('name',bio.name),
        bio_cv('name_description',('[NAME] '+bio.name+' [DESCRIPTION] '+bio.description))
    ]
    bio_results=pd.DataFrame(bio_exp)
    bio_results.to_csv(RESULTS_DIR/'biotools_external_validation.csv',index=False)
    bio[['id','name','description','label']].to_parquet(DATA_DIR/'biotools_top6_single_topic.parquet',index=False)
    display(bio_results.sort_values('f1_macro_mean',ascending=False))


## 10. Optional lightweight LLM pilot

This is **not required for the core paper**. It runs only after the robustness/generalizability experiments above. It evaluates a balanced subset using a public ~3B instruction model in 4-bit mode. Results should be presented as a pilot, not as the primary evidence.

If Colab assigned only CPU, this cell skips automatically.


In [ ]:
if RUN_LLM_PILOT:
    import torch
    if not torch.cuda.is_available():
        print('No GPU: skipping LLM pilot.')
    else:
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

        MODEL_ID='Qwen/Qwen2.5-3B-Instruct'
        qconf=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_compute_dtype=torch.float16)
        tok=AutoTokenizer.from_pretrained(MODEL_ID)
        llm=AutoModelForCausalLM.from_pretrained(
            MODEL_ID,device_map='auto',quantization_config=qconf,torch_dtype=torch.float16
        )

        # Prefer abstracts, then descriptions.
        source='abstract' if (single.abstract.str.len()>0).sum()>=300 else 'description'
        pilot=single[single[source].str.len()>0].copy()
        pilot['label']=pilot.labels.map(lambda x:x[0])
        pilot=pilot[pilot.label.isin(TARGET_LABELS)]

        # Balanced evaluation sample.
        n_each=min(50,pilot.label.value_counts().min())
        eval_df=(pilot.groupby('label',group_keys=False)
                      .apply(lambda x:x.sample(n=n_each,random_state=SEED))
                      .reset_index(drop=True))

        label_text='; '.join(TARGET_LABELS)
        def predict_one(text):
            prompt=f"""Classify the research software into exactly one of these categories:
{label_text}

Return only the category name, with no explanation.

Text:
{text[:5000]}
"""
            messages=[{'role':'user','content':prompt}]
            inp=tok.apply_chat_template(messages,add_generation_prompt=True,return_tensors='pt').to(llm.device)
            with torch.no_grad():
                out=llm.generate(inp,max_new_tokens=16,do_sample=False,pad_token_id=tok.eos_token_id)
            ans=tok.decode(out[0][inp.shape[-1]:],skip_special_tokens=True).strip()
            # Exact/contained canonical label normalization.
            for lab in TARGET_LABELS:
                if ans.lower()==lab.lower() or lab.lower() in ans.lower():
                    return lab
            return ans

        preds=[]
        for text in tqdm(eval_df[source],desc='LLM pilot'):
            preds.append(predict_one(text))
        eval_df['prediction']=preds
        eval_df.to_csv(RESULTS_DIR/'llm_pilot_predictions.csv',index=False)
        valid=eval_df.prediction.isin(TARGET_LABELS)
        llm_summary=pd.DataFrame([{
            'model':MODEL_ID,'source':source,'n':len(eval_df),
            'valid_output_rate':valid.mean(),
            'f1_macro':f1_score(eval_df.label,eval_df.prediction,labels=TARGET_LABELS,average='macro',zero_division=0),
            'f1_micro':f1_score(eval_df.label,eval_df.prediction,labels=TARGET_LABELS,average='micro',zero_division=0)
        }])
        llm_summary.to_csv(RESULTS_DIR/'llm_pilot_summary.csv',index=False)
        display(llm_summary)


## 11. Build paper-ready master tables

This cell joins the main checkpoints into compact CSV and LaTeX-friendly summaries. It never fabricates missing results: only completed checkpoint files are included.


In [ ]:
result_files=[
    'single_label_available_case.csv',
    'single_label_matched_individual.csv',
    'single_label_attribute_combinations.csv',
    'multilabel_tfidf.csv',
    'sbert_matched_results.csv',
    'biotools_external_validation.csv',
    'llm_pilot_summary.csv'
]

frames=[]
for fn in result_files:
    p=RESULTS_DIR/fn
    if p.exists():
        x=pd.read_csv(p)
        x.insert(0,'experiment_family',fn.replace('.csv',''))
        frames.append(x)

master=pd.concat(frames,ignore_index=True,sort=False) if frames else pd.DataFrame()
master.to_csv(RESULTS_DIR/'MASTER_RESULTS.csv',index=False)
display(master)

# Compact paper table: key mean/std values.
cols=[c for c in [
    'experiment_family','config','model','embedding','fields','n',
    'f1_macro_mean','f1_macro_std','f1_micro_mean','f1_micro_std',
    'f1_weighted_mean','accuracy_mean','subset_accuracy_mean','hamming_loss_mean',
    'f1_macro','f1_micro','valid_output_rate'
] if c in master.columns]
paper_table=master[cols].copy()
paper_table.to_csv(RESULTS_DIR/'PAPER_TABLE.csv',index=False)

try:
    latex=paper_table.round(3).to_latex(index=False,na_rep='--')
    (RESULTS_DIR/'PAPER_TABLE.tex').write_text(latex)
except Exception as e:
    print('LaTeX export warning:',e)

print('\nDONE.')
print('All outputs:',RESULTS_DIR)
print('Core file:',RESULTS_DIR/'MASTER_RESULTS.csv')
print('Paper table:',RESULTS_DIR/'PAPER_TABLE.csv')
